# Modelimi në Fizikë — Provim i pjesshëm II

**Koha në dispozicion:** 40 minuta  
**Totali:** 40 pikë  
**Emri dhe mbiemri:** ____________________________  
**Data:** ____________________________

Ky provim vlerëson aftësinë për të analizuar sisteme jolineare/kaotike, modele diskrete të popullsisë/epidemive dhe një algoritëm të thjeshtë renditjeje.


## Udhëzime

- Plotësoni vetëm pjesët e shënuara me `TODO` ose `...`.
- Mos ndryshoni emrat e funksioneve të dhëna.
- Figurat duhet të jenë të lexueshme dhe të etiketuara.
- Përgjigjet teorike duhet të lidhen me rezultatin numerik, jo vetëm me përkufizime të përgjithshme.
- Lejohet përdorimi i `numpy` dhe `matplotlib`. Nuk kërkohet `scipy`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


---
## Ushtrimi 1 — Harta logjistike dhe treguesi i Lyapunov-it (12 pikë)

Harta logjistike jepet nga

$$
x_{n+1}=r x_n(1-x_n), \qquad 0<x_0<1.
$$

Ajo është një shembull standard i kalimit nga sjellja periodike drejt kaosit për vlera të caktuara të parametrit \(r\).

**Pikëzimi:**

- 3 pikë: implementimi korrekt i hartës logjistike;
- 3 pikë: ndërtimi i diagramit të bifurkacionit;
- 3 pikë: implementimi i treguesit të Lyapunov-it;
- 3 pikë: interpretimi i rezultatit për \(r=3.2\) dhe \(r=3.9\).


In [ ]:
def logistic_step(x, r):
    # TODO 1.1: plotësoni hartën logjistike
    return ...


def logistic_orbit(r, x0=0.2, n_steps=1000, n_discard=200):
    x = float(x0)
    orbit = []

    for n in range(n_steps):
        x = logistic_step(x, r)
        if n >= n_discard:
            orbit.append(x)

    return np.array(orbit)


def lyapunov_logistic(r, x0=0.2, n_steps=2000, n_discard=500):
    x = float(x0)
    values = []

    for n in range(n_steps):
        x = logistic_step(x, r)
        if n >= n_discard:
            # Derivati i hartës: f'(x)=r(1-2x)
            derivative = ...  # TODO 1.2
            values.append(np.log(abs(derivative) + 1e-12))

    # TODO 1.3: ktheni mesataren e vlerave logaritmike
    return ...


In [ ]:
# Diagram bifurkacioni
r_values = np.linspace(2.8, 4.0, 350)
R_plot = []
X_plot = []

for r in r_values:
    xs = logistic_orbit(r, x0=0.2, n_steps=900, n_discard=700)
    R_plot.extend([r] * len(xs))
    X_plot.extend(xs)

plt.figure(figsize=(8, 4.5))
plt.plot(R_plot, X_plot, ",", alpha=0.5)
plt.xlabel("r")
plt.ylabel("vlerat afatgjata të x")
plt.title("Diagrami i bifurkacionit për hartën logjistike")
plt.show()

for r in [3.2, 3.9]:
    lam = lyapunov_logistic(r)
    print(f"r = {r:.1f}, lambda ≈ {lam:.4f}")


**Përgjigje e shkurtër për Ushtrimin 1:**  
Duke u bazuar te diagrami dhe te shenja e treguesit të Lyapunov-it, përshkruani ndryshimin midis \(r=3.2\) dhe \(r=3.9\).

_Përgjigjja juaj:_


---
## Ushtrimi 2 — Sistemi i Lorencit dhe ndjeshmëria ndaj kushteve fillestare (10 pikë)

Sistemi i Lorencit jepet nga

$$
\dot{x}=\sigma(y-x),
\qquad
\dot{y}=x(\rho-z)-y,
\qquad
\dot{z}=xy-\beta z.
$$

**Detyra:** Plotësoni anën e djathtë të sistemit dhe tregoni numerikisht se dy kushte fillestare shumë të afërta largohen me kalimin e kohës.

**Pikëzimi:**

- 4 pikë: formulimi i saktë i sistemit;
- 3 pikë: simulimi i dy trajektoreve;
- 3 pikë: interpretimi i rritjes së distancës në hapësirën e fazës.


In [ ]:
def rhs_lorenz(state, sigma=10.0, rho=28.0, beta=8/3):
    x, y, z = state

    # TODO 2.1: plotësoni ekuacionet e Lorencit
    dxdt = ...
    dydt = ...
    dzdt = ...
    return np.array([dxdt, dydt, dzdt], dtype=float)


def rk4_step(rhs, state, dt, *args, **kwargs):
    k1 = rhs(state, *args, **kwargs)
    k2 = rhs(state + 0.5 * dt * k1, *args, **kwargs)
    k3 = rhs(state + 0.5 * dt * k2, *args, **kwargs)
    k4 = rhs(state + dt * k3, *args, **kwargs)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def integrate_lorenz(state0, t_end=25.0, dt=0.01):
    t = np.arange(0.0, t_end + dt, dt)
    y = np.zeros((len(t), 3), dtype=float)
    y[0] = np.array(state0, dtype=float)

    for n in range(len(t) - 1):
        y[n + 1] = rk4_step(rhs_lorenz, y[n], dt)

    return t, y


In [ ]:
state_a = np.array([1.0, 1.0, 1.0])
state_b = np.array([1.0 + 1e-6, 1.0, 1.0])

t, ya = integrate_lorenz(state_a)
_, yb = integrate_lorenz(state_b)

# TODO 2.2: llogaritni distancën Euklidiane midis dy trajektoreve në çdo kohë
distance = ...

plt.figure()
plt.semilogy(t, distance)
plt.xlabel("t")
plt.ylabel("distanca midis trajektoreve")
plt.title("Ndjeshmëria ndaj kushteve fillestare në sistemin e Lorencit")
plt.show()

plt.figure()
plt.plot(ya[:, 0], ya[:, 2], label="trajektorja A")
plt.plot(yb[:, 0], yb[:, 2], alpha=0.75, label="trajektorja B")
plt.xlabel("x")
plt.ylabel("z")
plt.title("Projeksioni x-z i dy trajektoreve të Lorencit")
plt.legend()
plt.show()


**Përgjigje e shkurtër për Ushtrimin 2:**  
A nënkupton kaosi se sistemi është i rastësishëm? Shpjegoni shkurt dallimin midis determinizmit dhe parashikueshmërisë afatgjatë.

_Përgjigjja juaj:_


---
## Ushtrimi 3 — Modeli SIR për përhapjen epidemike (10 pikë)

Modeli SIR përshkruhet nga

$$
\dot{S}=-\beta SI,
\qquad
\dot{I}=\beta SI-\gamma I,
\qquad
\dot{R}=\gamma I.
$$

Në këtë ushtrim, \(S\), \(I\), \(R\) janë fraksione të popullsisë dhe \(S+I+R\approx1\).

**Pikëzimi:**

- 4 pikë: implementimi i saktë i anës së djathtë;
- 2 pikë: ruajtja numerike e shumës \(S+I+R\);
- 2 pikë: gjetja e pikut të infektimeve;
- 2 pikë: interpretimi i rolit të \(\beta\) dhe \(\gamma\).


In [ ]:
def rhs_sir(state, beta=0.35, gamma=0.10):
    S, I, R = state

    # TODO 3.1: plotësoni ekuacionet SIR
    dSdt = ...
    dIdt = ...
    dRdt = ...
    return np.array([dSdt, dIdt, dRdt], dtype=float)


def integrate_sir(state0=(0.99, 0.01, 0.0), beta=0.35, gamma=0.10, t_end=120.0, dt=0.1):
    t = np.arange(0.0, t_end + dt, dt)
    y = np.zeros((len(t), 3), dtype=float)
    y[0] = np.array(state0, dtype=float)

    for n in range(len(t) - 1):
        # Metoda Euler është e mjaftueshme për këtë ushtrim.
        y[n + 1] = y[n] + dt * rhs_sir(y[n], beta=beta, gamma=gamma)

    return t, y


In [ ]:
t, y = integrate_sir()
S, I, R = y[:, 0], y[:, 1], y[:, 2]

# TODO 3.2: llogaritni shumën S+I+R dhe gabimin maksimal nga 1
total_population = ...
max_mass_error = ...

# TODO 3.3: gjeni kohën dhe vlerën e pikut të I(t)
peak_index = ...
t_peak = ...
I_peak = ...

print("Gabimi maksimal në S+I+R:", max_mass_error)
print("Piku i I(t):", I_peak, "në t =", t_peak)

plt.figure()
plt.plot(t, S, label="S")
plt.plot(t, I, label="I")
plt.plot(t, R, label="R")
plt.xlabel("t")
plt.ylabel("fraksioni i popullsisë")
plt.title("Modeli SIR")
plt.legend()
plt.show()


**Përgjigje e shkurtër për Ushtrimin 3:**  
Çfarë pritet të ndodhë me pikun e \(I(t)\) nëse \(\beta\) zvogëlohet, duke mbajtur \(\gamma\) të pandryshuar? Argumentoni fizikisht.

_Përgjigjja juaj:_


---
## Ushtrimi 4 — Renditje e nyjeve me një algoritëm të tipit PageRank (8 pikë)

Merrni grafikun e mëposhtëm të orientuar. Elementi \(A_{ij}=1\) tregon se faqja/nyja \(j\) lidh drejt faqes/nyjes \(i\). Pra, kolonat përfaqësojnë nyjet burim.

**Pikëzimi:**

- 3 pikë: normalizimi korrekt i matricës së tranzicionit;
- 3 pikë: iterimi me faktor zbutjeje;
- 2 pikë: interpretimi i nyjes me renditjen më të lartë.


In [ ]:
A = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 0, 1],
    [0, 0, 0, 1, 0],
    [0, 0, 1, 0, 1],
    [1, 0, 0, 1, 0],
], dtype=float)

labels = ["A", "B", "C", "D", "E"]


def build_transition_matrix(A):
    # Kthen matricën kolonë-stokastike të tranzicionit.
    M = A.copy().astype(float)
    n = M.shape[0]

    for j in range(n):
        col_sum = M[:, j].sum()

        # TODO 4.1: nëse kolona ka dalje, normalizojeni; nëse jo, shpërndani njëtrajtshëm
        if col_sum > 0:
            M[:, j] = ...
        else:
            M[:, j] = ...

    return M


def pagerank(A, damping=0.85, n_iter=100):
    n = A.shape[0]
    M = build_transition_matrix(A)
    p = np.ones(n) / n

    for _ in range(n_iter):
        # TODO 4.2: plotësoni iterimin PageRank
        p = ...

    return p

scores = pagerank(A)

for label, score in zip(labels, scores):
    print(f"{label}: {score:.4f}")

# TODO 4.3: gjeni nyjen me pikën më të lartë
best_node = ...
print("Nyja me renditjen më të lartë:", best_node)

plt.figure()
plt.bar(labels, scores)
plt.xlabel("nyja")
plt.ylabel("PageRank")
plt.title("Renditja e nyjeve")
plt.show()


**Përgjigje e shkurtër për Ushtrimin 4:**  
Pse nyja me më shumë lidhje hyrëse nuk është domosdoshmërisht gjithmonë nyja me PageRank më të lartë?

_Përgjigjja juaj:_


---
## Kontroll final para dorëzimit

Para se ta dorëzoni notebook-un:

- kontrolloni që qelizat të ekzekutohen në rend nga lart-poshtë;
- figurat të jenë të qarta;
- përgjigjet teorike të jenë plotësuar;
- emrat e funksioneve të mos jenë ndryshuar.
